# 🚀 PIPELINE ETL: XÂY DỰNG 2 BẢNG FACT (SILVER → GOLD)
## 📌 BẢNG DANH MỤC CHI TIẾT CÁC CỘT, KIỂU DỮ LIỆU & KHÓA (CHUẨN HÓA MỚI)

### 1. `Fact_Sales_Order_Items` (17 cột - PK: `Sales_Item_Key`)
| STT | Tên cột | Kiểu dữ liệu | Nullable | Loại khóa | Ý nghĩa nghiệp vụ |
| :---: | :--- | :--- | :---: | :---: | :--- |
| 1 | `Sales_Item_Key` | `BIGINT (Identity)` | NO | **PK** | Khóa thay thế duy nhất cho từng dòng sản phẩm bán ra |
| 2 | `Order_Date_Key` | `INT` | NO | **FK** | Nối sang `Dim_Date(Date_Key)` định dạng YYYYMMDD |
| 3 | `Customer_ID` | `INT` | NO | **FK** | Nối sang `Dim_Customer(Customer_ID)` |
| 4 | `Product_ID` | `INT` | NO | **FK** | Nối sang `Dim_Product(Product_ID)` |
| 5 | `Promo_ID` | `NVARCHAR(100)` | YES | **FK** | Nối sang `Dim_Promotion(Promo_ID)` |
| 6 | `Zip_Code` | `NVARCHAR(50)` | YES | **FK** | Nối sang `Dim_Geography(Zip_Code)` |
| 7 | `Sales_Employee_ID` | `NVARCHAR(50)` | YES | **FK** | Nối sang `Dim_Sales_Employee(Sales_Employee_ID)` |
| 8 | `Order_ID` | `INT` | NO | - | Mã đơn hàng tự nhiên (Degenerate Dimension) |
| 9 | `Order_Source` | `NVARCHAR(50)` | YES | - | Nguồn đơn (Website, App, In-Store...) |
| 10 | `Order_Status` | `NVARCHAR(50)` | YES | - | Trạng thái đơn (delivered, shipped, cancelled...) |
| 11 | `Device_Type` | `NVARCHAR(50)` | YES | - | Thiết bị mua hàng (Mobile, Desktop, Tablet) |
| 12 | `Quantity` | `INT` | NO | - | Số lượng sản phẩm mua (> 0) |
| 13 | `Unit_Price` | `DECIMAL(18,2)` | NO | - | Đơn giá niêm yết của sản phẩm |
| 14 | `Discount_Amount` | `DECIMAL(18,2)` | NO | - | Tiền giảm giá trên từng dòng sản phẩm |
| 15 | `Line_Total` | `DECIMAL(18,2)` | NO | - | Doanh thu thực tế (`Quantity * Unit_Price - Discount_Amount`) |
| 16 | `COGS` | `DECIMAL(18,2)` | YES | - | Giá vốn hàng bán (`Quantity * Standard_Cost`) |
| 17 | `Gross_Profit` | `DECIMAL(18,2)` | YES | - | Lợi nhuận gộp (`Line_Total - COGS`) |

### 2. `Fact_Customer_Metrics` (17 cột - PK: `Customer_Metric_Key`)
| STT | Tên cột | Kiểu dữ liệu | Nullable | Loại khóa | Ý nghĩa nghiệp vụ |
| :---: | :--- | :--- | :---: | :---: | :--- |
| 1 | `Customer_Metric_Key` | `BIGINT (Identity)` | NO | **PK** | Khóa thay thế bản ghi tổng hợp khách hàng |
| 2 | `Customer_ID` | `INT` | NO | **FK / UQ** | Nối sang `Dim_Customer(Customer_ID)` |
| 3 | `Snapshot_Date_Key` | `INT` | NO | **FK / UQ** | Ngày chốt số liệu snapshot định dạng YYYYMMDD |
| 4 | `Zip_Code` | `NVARCHAR(50)` | YES | **FK** | Nối sang `Dim_Geography(Zip_Code)` |
| 5 | `First_Order_Date` | `DATE` | YES | - | Ngày đầu tiên khách đặt đơn delivered (YYYY-MM-DD) |
| 6 | `Last_Order_Date` | `DATE` | YES | - | Ngày gần nhất khách đặt đơn delivered (YYYY-MM-DD) |
| 7 | `Total_Delivered_Orders` | `INT` | NO | - | Tổng số đơn hàng giao thành công |
| 8 | `Total_Lifetime_Revenue` | `DECIMAL(18,2)` | NO | - | Tổng doanh thu tích lũy toàn vòng đời |
| 9 | `Total_Lifetime_Items` | `INT` | NO | - | Tổng số sản phẩm đã mua qua các đơn delivered |
| 10 | `Lifespan_Days` | `INT` | YES | - | Vòng đời khách hàng tính bằng số ngày (`Last_Date - First_Date`) |
| 11 | `Inter_Purchase_Days` | `DECIMAL(10,2)` | YES | - | Khoảng cách trung bình giữa các lần mua (PS5) |
| 12 | `Customer_AOV` | `DECIMAL(18,2)` | YES | - | Giá trị trung bình đơn hàng (`Lifetime_Revenue / Orders`) |
| 13 | `Customer_Basket_Size` | `DECIMAL(10,2)` | YES | - | Số sản phẩm trung bình mỗi đơn (`Lifetime_Items / Orders`) |
| 14 | `Is_Repeat_Customer` | `BIT` | NO | - | 1: Khách mua lại (Orders >= 2), 0: Mua 1 lần |
| 15 | `Customer_Type` | `NVARCHAR(50)` | NO | - | Phân loại: 'Repeat Customer' / 'One-time Buyer' |
| 16 | `CLV_Rank` | `INT` | YES | - | Xếp hạng giá trị vòng đời khách hàng (PS2) |
| 17 | `Customer_Tier` | `NVARCHAR(50)` | YES | - | Phân hạng: VIP Customers, High-value, Potential, Low-value |

## 🛠️ BƯỚC 1: Cấu hình Môi trường & Tự động Quét Tìm / Giải Nén File CSV

In [1]:
import os
import sys
import glob
import zipfile
import shutil
import pandas as pd
import numpy as np

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Thư mục xuất dữ liệu Fact
OUTPUT_DIR = 'Gold_Facts'
OUTPUT_DW_DIR = 'Gold_Data_Warehouse'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DW_DIR, exist_ok=True)

# 1. Tự động giải nén file zip nếu có
search_roots = ['.', 'Silver data', '/content', '/content/drive/MyDrive']
for root in search_roots:
    if os.path.exists(root):
        for zip_path in glob.glob(os.path.join(root, '*.zip')):
            if 'Gold' not in os.path.basename(zip_path):
                print(f"📦 Phát hiện file nén: [{zip_path}]. Đang tự động giải nén...")
                try:
                    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                        zip_ref.extractall('.')
                    print(f"✓ Giải nén thành công [{zip_path}]!")
                except Exception as e:
                    print(f"⚠️ Không thể giải nén {zip_path}: {e}")

# 2. Hàm tìm kiếm file đệ quy thông minh
def find_and_read_csv(filename, required=True):
    target_name = os.path.basename(filename).lower()
    candidate_files = []
    for search_dir in ['.', 'Silver data', '/content', '..']:
        if os.path.exists(search_dir):
            for root, dirs, files in os.walk(search_dir):
                if OUTPUT_DIR in root or OUTPUT_DW_DIR in root or '.git' in root or '.ipynb_checkpoints' in root:
                    continue
                for f in files:
                    if f.lower() == target_name:
                        candidate_files.append(os.path.join(root, f))

    if not candidate_files:
        if required:
            raise FileNotFoundError(f"❌ KHÔNG TÌM THẤY FILE '{filename}'! Vui lòng upload vào môi trường làm việc.")
        return None

    found_path = candidate_files[0]
    try:
        df = pd.read_csv(found_path, sep=',', low_memory=False)
    except Exception:
        df = pd.read_csv(found_path, sep=';', low_memory=False)

    df.columns = df.columns.str.replace('\ufeff', '').str.strip()
    print(f"✓ Đã đọc [{filename}] từ: [{found_path}] -> ({df.shape[0]:,} dòng, {df.shape[1]} cột)")
    return df

print("\n🔥 HỆ THỐNG ĐÃ SẴN SÀNG ĐỌC DỮ LIỆU SILVER!")


🔥 HỆ THỐNG ĐÃ SẴN SÀNG ĐỌC DỮ LIỆU SILVER!


## 📦 BƯỚC 2: Đọc Dữ liệu Silver và Khởi tạo Danh mục Khóa Hợp lệ (Conformed Natural Keys)

In [2]:
# 1. Đọc dữ liệu Silver gốc
df_orders = find_and_read_csv('orders.csv')
df_items = find_and_read_csv('Order_items.csv')
df_products = find_and_read_csv('silver_product.csv')
df_customers = find_and_read_csv('customers_silver.csv')
df_geo = find_and_read_csv('geography.csv')
df_promo = find_and_read_csv('promotions.csv', required=False)
df_emp = find_and_read_csv('sale_employee.csv', required=False)

print("\n--- KHỞI TẠO TẬP HỢP KHÓA TỰ NHIÊN (NATURAL KEYS) & BẢN ĐỒ LIÊN KẾT ---")

# 1. Dim_Geography (Zip_Code)
df_geo_clean = df_geo.drop_duplicates(subset=['zip']).copy()
df_geo_clean['zip'] = df_geo_clean['zip'].astype(str).str.strip()
valid_zips = set(df_geo_clean['zip'])
print(f"✓ Dim_Geography: {len(valid_zips):,} mã bưu chính hợp lệ")

# 2. Dim_Customer (Customer_ID -> Zip_Code)
df_cust_clean = df_customers.dropna(subset=['customer_id']).drop_duplicates(subset=['customer_id']).copy()
df_cust_clean['customer_id'] = pd.to_numeric(df_cust_clean['customer_id'], errors='coerce').astype(int)
df_cust_clean['zip'] = df_cust_clean['zip'].astype(str).str.strip()
df_cust_clean['Zip_Code'] = np.where(df_cust_clean['zip'].isin(valid_zips), df_cust_clean['zip'], None)
valid_customers = set(df_cust_clean['customer_id'])
cust_zip_map = dict(zip(df_cust_clean['customer_id'], df_cust_clean['Zip_Code']))
print(f"✓ Dim_Customer: {len(valid_customers):,} khách hàng hợp lệ")

# 3. Dim_Product (Product_ID -> cogs / Standard_Cost)
df_prod_clean = df_products.dropna(subset=['product_id']).drop_duplicates(subset=['product_id']).copy()
df_prod_clean['product_id'] = pd.to_numeric(df_prod_clean['product_id'], errors='coerce').astype(int)
df_prod_clean['cogs'] = pd.to_numeric(df_prod_clean['cogs'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0.0).round(2)
valid_products = set(df_prod_clean['product_id'])
prod_cogs_map = dict(zip(df_prod_clean['product_id'], df_prod_clean['cogs']))
print(f"✓ Dim_Product: {len(valid_products):,} sản phẩm hợp lệ")

# 4. Dim_Promotion (Promo_ID)
if df_promo is not None:
    df_promo_clean = df_promo.dropna(subset=['promo_id']).drop_duplicates(subset=['promo_id']).copy()
    df_promo_clean['promo_id'] = df_promo_clean['promo_id'].astype(str).str.strip()
    valid_promos = set(df_promo_clean['promo_id'])
else:
    valid_promos = set()
print(f"✓ Dim_Promotion: {len(valid_promos):,} chương trình khuyến mãi hợp lệ")

# 5. Dim_Sales_Employee (Sales_Employee_ID)
if df_emp is not None:
    df_emp_clean = df_emp.dropna(subset=['sales_employee_id']).drop_duplicates(subset=['sales_employee_id']).copy()
    df_emp_clean['sales_employee_id'] = df_emp_clean['sales_employee_id'].astype(str).str.strip()
    valid_employees = set(df_emp_clean['sales_employee_id'])
else:
    valid_employees = set()
print(f"✓ Dim_Sales_Employee: {len(valid_employees):,} nhân viên hợp lệ")

✓ Đã đọc [orders.csv] từ: [./orders.csv] -> (646,945 dòng, 9 cột)
✓ Đã đọc [Order_items.csv] từ: [./Order_items.csv] -> (714,653 dòng, 7 cột)
✓ Đã đọc [silver_product.csv] từ: [./silver_product.csv] -> (2,412 dòng, 8 cột)
✓ Đã đọc [customers_silver.csv] từ: [./customers_silver.csv] -> (121,930 dòng, 6 cột)
✓ Đã đọc [geography.csv] từ: [./geography.csv] -> (39,948 dòng, 4 cột)
✓ Đã đọc [promotions.csv] từ: [./promotions.csv] -> (50 dòng, 10 cột)
✓ Đã đọc [sale_employee.csv] từ: [./sale_employee.csv] -> (200 dòng, 5 cột)

--- KHỞI TẠO TẬP HỢP KHÓA TỰ NHIÊN (NATURAL KEYS) & BẢN ĐỒ LIÊN KẾT ---
✓ Dim_Geography: 39,948 mã bưu chính hợp lệ
✓ Dim_Customer: 121,930 khách hàng hợp lệ
✓ Dim_Product: 2,412 sản phẩm hợp lệ
✓ Dim_Promotion: 50 chương trình khuyến mãi hợp lệ
✓ Dim_Sales_Employee: 200 nhân viên hợp lệ


## 🛒 BƯỚC 3: Xây dựng Bảng `Fact_Sales_Order_Items` (Transactional Fact - 17 Cột Chuẩn)
* **Khóa chính (PK):** `Sales_Item_Key` (Surrogate Identity tự tăng)
* **Khóa ngoại (FK):** `Order_Date_Key`, `Customer_ID`, `Product_ID`, `Promo_ID`, `Zip_Code`, `Sales_Employee_ID`
* **Degenerate & Dimension Attributes:** `Order_ID`, `Order_Source`, `Order_Status`, `Device_Type`
* **Measures:** `Quantity`, `Unit_Price`, `Discount_Amount`, `Line_Total`, `COGS`, `Gross_Profit`

In [3]:
print("=== XÂY DỰNG FACT_SALES_ORDER_ITEMS ===")

# 1. Chuẩn hóa Order Items
df_items_clean = df_items.copy()
df_items_clean['order_id'] = pd.to_numeric(df_items_clean['order_id'], errors='coerce').astype(int)
df_items_clean['product_id'] = pd.to_numeric(df_items_clean['product_id'], errors='coerce').astype(int)
df_items_clean['quantity'] = pd.to_numeric(df_items_clean['quantity'], errors='coerce').fillna(1).astype(int)
df_items_clean['unit_price'] = pd.to_numeric(df_items_clean['unit_price'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0.0).round(2)
df_items_clean['discount_amount'] = pd.to_numeric(df_items_clean['discount_amount'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0.0).round(2)
df_items_clean['promo_id_raw'] = df_items_clean['promo_id'].fillna('').astype(str).str.strip()
df_items_clean['Promo_ID'] = np.where(df_items_clean['promo_id_raw'].isin(valid_promos), df_items_clean['promo_id_raw'], None)

# 2. Chuẩn hóa Orders
df_orders_clean = df_orders.copy()
df_orders_clean['order_id'] = pd.to_numeric(df_orders_clean['order_id'], errors='coerce').astype(int)
df_orders_clean['customer_id'] = pd.to_numeric(df_orders_clean['customer_id'], errors='coerce').astype(int)
df_orders_clean['order_date_dt'] = pd.to_datetime(df_orders_clean['order_date'], errors='coerce')
df_orders_clean['Order_Date_Key'] = df_orders_clean['order_date_dt'].dt.strftime('%Y%m%d').fillna(19000101).astype(int)
df_orders_clean['emp_id_raw'] = df_orders_clean['sales_employee_id'].astype(str).str.strip()
df_orders_clean['Sales_Employee_ID'] = np.where(df_orders_clean['emp_id_raw'].isin(valid_employees), df_orders_clean['emp_id_raw'], None)
df_orders_clean['order_source'] = df_orders_clean['order_source'].fillna('').astype(str).str.strip().replace({'nan': None, 'none': None, '': None})
df_orders_clean['device_type'] = df_orders_clean['device_type'].fillna('').astype(str).str.strip().replace({'nan': None, 'none': None, '': None})
df_orders_clean['order_status'] = df_orders_clean['order_status'].fillna('delivered').astype(str).str.strip().replace({'nan': None, 'none': None, '': None})

# Chuẩn hóa Zip_Code từ orders (nếu có) hoặc map theo customer
if 'zip' in df_orders_clean.columns:
    df_orders_clean['order_zip_clean'] = df_orders_clean['zip'].astype(str).str.strip()
    df_orders_clean['order_zip_valid'] = np.where(df_orders_clean['order_zip_clean'].isin(valid_zips), df_orders_clean['order_zip_clean'], None)
else:
    df_orders_clean['order_zip_valid'] = None

# 3. Merge Items với Orders (Inner Join)
fact_sales = df_items_clean.merge(
    df_orders_clean[['order_id', 'customer_id', 'Order_Date_Key', 'Sales_Employee_ID', 'order_source', 'order_status', 'device_type', 'order_zip_valid']],
    on='order_id',
    how='inner'
)

fact_sales['Customer_ID'] = fact_sales['customer_id']
fact_sales['Product_ID'] = fact_sales['product_id']

# Đảm bảo toàn vẹn khóa ngoại sang Dim_Geography
fact_sales['cust_zip'] = fact_sales['Customer_ID'].map(cust_zip_map)
fact_sales['Zip_Code'] = fact_sales['order_zip_valid'].fillna(fact_sales['cust_zip'])

# Lọc đảm bảo toàn vẹn khóa ngoại sang Dim_Product và Dim_Customer
fact_sales = fact_sales[fact_sales['Product_ID'].isin(valid_products)].copy()
fact_sales = fact_sales[fact_sales['Customer_ID'].isin(valid_customers)].copy()

# 4. Tính toán Measures theo đúng công thức tài chính
unit_cogs = fact_sales['Product_ID'].map(prod_cogs_map).fillna(0.0).round(2)
fact_sales['Line_Total'] = (fact_sales['quantity'] * fact_sales['unit_price'] - fact_sales['discount_amount']).round(2)
fact_sales['COGS'] = (fact_sales['quantity'] * unit_cogs).round(2)
fact_sales['Gross_Profit'] = (fact_sales['Line_Total'] - fact_sales['COGS']).round(2)

# 5. Sinh Khóa chính Surrogate Key tự tăng (Sales_Item_Key)
fact_sales.reset_index(drop=True, inplace=True)
fact_sales['Sales_Item_Key'] = fact_sales.index + 1

# 6. Đổi tên và sắp xếp đúng 100% theo DDL SQL Schema (17 cột)
fact_sales = fact_sales.rename(columns={
    'order_id': 'Order_ID',
    'order_source': 'Order_Source',
    'order_status': 'Order_Status',
    'device_type': 'Device_Type',
    'quantity': 'Quantity',
    'unit_price': 'Unit_Price',
    'discount_amount': 'Discount_Amount'
})

cols_fact_sales_target = [
    'Sales_Item_Key', 'Order_Date_Key', 'Customer_ID', 'Product_ID', 'Promo_ID', 'Zip_Code',
    'Sales_Employee_ID', 'Order_ID', 'Order_Source', 'Order_Status', 'Device_Type',
    'Quantity', 'Unit_Price', 'Discount_Amount', 'Line_Total', 'COGS', 'Gross_Profit'
]
fact_sales = fact_sales[cols_fact_sales_target]

print(f"✓ Xây dựng Fact_Sales_Order_Items hoàn tất! Tổng số giao dịch dòng đơn: {len(fact_sales):,}")
display(fact_sales.head(5))

=== XÂY DỰNG FACT_SALES_ORDER_ITEMS ===
✓ Xây dựng Fact_Sales_Order_Items hoàn tất! Tổng số giao dịch dòng đơn: 714,653


,Sales_Item_Key,Order_Date_Key,Customer_ID,Product_ID,Promo_ID,Zip_Code,Sales_Employee_ID,Order_ID,Order_Source,Order_Status,Device_Type,Quantity,Unit_Price,Discount_Amount,Line_Total,COGS,Gross_Profit
0,1,20120704,58578,2400,None,1109,EMP0103,1,paid_search,delivered,desktop,7,1138.22,0.00,7967.54,7376.60,590.94
1,2,20120704,58621,609,None,1330,EMP0180,2,paid_search,returned,mobile,7,10166.25,0.00,71163.75,62913.90,8249.85
2,3,20120704,58811,396,None,1473,EMP0093,3,direct,delivered,desktop,3,11220.33,0.00,33660.99,30273.03,3387.96
3,4,20120704,59453,635,None,2360,EMP0015,4,referral,delivered,desktop,5,10639.25,0.00,53196.25,46027.15,7169.10
4,5,20120706,57821,1935,None,2886,EMP0107,6,email_campaign,delivered,mobile,1,1597.84,0.00,1597.84,1048.70,549.14


## 👥 BƯỚC 4: Xây dựng Bảng `Fact_Customer_Metrics` (Customer Snapshot Fact - 17 Cột Chuẩn)
* **Khóa chính (PK):** `Customer_Metric_Key` (Surrogate Identity tự tăng)
* **Khóa ngoại / Ràng buộc Unique:** `Customer_ID`, `Snapshot_Date_Key`, `Zip_Code`
* **Chỉ số hành vi & Vòng đời:** `First_Order_Date`, `Last_Order_Date`, `Total_Delivered_Orders`, `Total_Lifetime_Revenue`, `Total_Lifetime_Items`, `Lifespan_Days`, `Inter_Purchase_Days`, `Customer_AOV`, `Customer_Basket_Size`
* **Phân khúc nâng cao (PS1, PS2):** `Is_Repeat_Customer`, `Customer_Type`, `CLV_Rank`, `Customer_Tier`

In [4]:
print("=== XÂY DỰNG FACT_CUSTOMER_METRICS ===")

# 1. Lọc đơn hàng hợp lệ đã giao thành công (delivered) để tính toán hành vi thực tế
df_sales_deliv = fact_sales[fact_sales['Order_Status'].str.lower() == 'delivered'].copy()
df_sales_deliv = df_sales_deliv.merge(df_orders_clean[['order_id', 'order_date_dt']], left_on='Order_ID', right_on='order_id', how='left')

# Ngày chốt số liệu snapshot (Lấy ngày lớn nhất trong dataset)
snapshot_dt = df_orders_clean['order_date_dt'].max()
snapshot_date_key = int(snapshot_dt.strftime('%Y%m%d'))
print(f"  -> Snapshot Date Key: {snapshot_date_key} ({snapshot_dt.strftime('%Y-%m-%d')})")

# 2. Tổng hợp chỉ số theo từng Khách hàng có đơn hàng delivered
cust_grp = df_sales_deliv.groupby('Customer_ID')
df_cust_agg = cust_grp.agg(
    Total_Lifetime_Revenue=('Line_Total', 'sum'),
    Total_Lifetime_Items=('Quantity', 'sum'),
    Total_Delivered_Orders=('Order_ID', 'nunique'),
    First_Order_Date_dt=('order_date_dt', 'min'),
    Last_Order_Date_dt=('order_date_dt', 'max')
).reset_index()

df_cust_agg['First_Order_Date'] = df_cust_agg['First_Order_Date_dt'].dt.strftime('%Y-%m-%d')
df_cust_agg['Last_Order_Date'] = df_cust_agg['Last_Order_Date_dt'].dt.strftime('%Y-%m-%d')
df_cust_agg['Total_Lifetime_Revenue'] = df_cust_agg['Total_Lifetime_Revenue'].round(2)
df_cust_agg['Lifespan_Days'] = (df_cust_agg['Last_Order_Date_dt'] - df_cust_agg['First_Order_Date_dt']).dt.days
df_cust_agg['Inter_Purchase_Days'] = np.where(
    df_cust_agg['Total_Delivered_Orders'] > 1,
    (df_cust_agg['Lifespan_Days'] / (df_cust_agg['Total_Delivered_Orders'] - 1)).round(2),
    0.0
)
df_cust_agg['Customer_AOV'] = (df_cust_agg['Total_Lifetime_Revenue'] / df_cust_agg['Total_Delivered_Orders']).round(2)
df_cust_agg['Customer_Basket_Size'] = (df_cust_agg['Total_Lifetime_Items'] / df_cust_agg['Total_Delivered_Orders']).round(2)

# 3. Phân khúc khách hàng (Repeat Customer vs One-time Buyer)
df_cust_agg['Is_Repeat_Customer'] = np.where(df_cust_agg['Total_Delivered_Orders'] > 1, 1, 0)
df_cust_agg['Customer_Type'] = np.where(df_cust_agg['Total_Delivered_Orders'] > 1, 'Repeat Customer', 'One-time Buyer')

# 4. Xếp hạng CLV & Phân hạng Tier (PS2: VIP, High-value, Potential, Low-value)
df_cust_agg['CLV_Rank'] = df_cust_agg['Total_Lifetime_Revenue'].rank(ascending=False, method='min').astype(int)
p95 = df_cust_agg['Total_Lifetime_Revenue'].quantile(0.95)
p80 = df_cust_agg['Total_Lifetime_Revenue'].quantile(0.80)
p50 = df_cust_agg['Total_Lifetime_Revenue'].quantile(0.50)

def get_tier(rev):
    if rev >= p95:
        return 'VIP Customers'
    elif rev >= p80:
        return 'High-value Customers'
    elif rev >= p50:
        return 'Potential Customers'
    else:
        return 'Low-value Customers'

df_cust_agg['Customer_Tier'] = df_cust_agg['Total_Lifetime_Revenue'].apply(get_tier)

# 5. Thiết lập Fact_Customer_Metrics (Chỉ giữ khách hàng có đơn hàng giao thành công: Total_Delivered_Orders >= 1)
# Lọc bỏ những người có Total_Delivered_Orders = 0 để tránh làm sai lệch phân khúc và các chỉ số KPI
fact_cust_metrics = df_cust_agg[df_cust_agg['Total_Delivered_Orders'] >= 1].copy()
fact_cust_metrics['Snapshot_Date_Key'] = snapshot_date_key
fact_cust_metrics['Zip_Code'] = fact_cust_metrics['Customer_ID'].map(cust_zip_map)

# Đảm bảo kiểu dữ liệu chuẩn xác
fact_cust_metrics['Total_Delivered_Orders'] = fact_cust_metrics['Total_Delivered_Orders'].astype(int)
fact_cust_metrics['Total_Lifetime_Revenue'] = fact_cust_metrics['Total_Lifetime_Revenue'].round(2)
fact_cust_metrics['Total_Lifetime_Items'] = fact_cust_metrics['Total_Lifetime_Items'].astype(int)
fact_cust_metrics['Lifespan_Days'] = fact_cust_metrics['Lifespan_Days'].astype(int)
fact_cust_metrics['Inter_Purchase_Days'] = fact_cust_metrics['Inter_Purchase_Days'].round(2)
fact_cust_metrics['Customer_AOV'] = fact_cust_metrics['Customer_AOV'].round(2)
fact_cust_metrics['Customer_Basket_Size'] = fact_cust_metrics['Customer_Basket_Size'].round(2)
fact_cust_metrics['Is_Repeat_Customer'] = fact_cust_metrics['Is_Repeat_Customer'].astype(int)
fact_cust_metrics['CLV_Rank'] = fact_cust_metrics['CLV_Rank'].astype(int)

# Sinh Khóa chính Surrogate Key tự tăng (Customer_Metric_Key)
fact_cust_metrics.sort_values(by='Customer_ID', inplace=True)
fact_cust_metrics.reset_index(drop=True, inplace=True)
fact_cust_metrics['Customer_Metric_Key'] = fact_cust_metrics.index + 1

# 6. Sắp xếp đúng chuẩn DDL SQL (17 cột)
cols_cust_target = [
    'Customer_Metric_Key', 'Customer_ID', 'Snapshot_Date_Key', 'Zip_Code',
    'First_Order_Date', 'Last_Order_Date',
    'Total_Delivered_Orders', 'Total_Lifetime_Revenue', 'Total_Lifetime_Items',
    'Lifespan_Days', 'Inter_Purchase_Days', 'Customer_AOV', 'Customer_Basket_Size',
    'Is_Repeat_Customer', 'Customer_Type',
    'CLV_Rank', 'Customer_Tier'
]
fact_cust_metrics = fact_cust_metrics[cols_cust_target]

print(f"✓ Xây dựng Fact_Customer_Metrics hoàn tất! Tổng số khách hàng có đơn giao thành công: {len(fact_cust_metrics):,}")
display(fact_cust_metrics.head(5))

=== XÂY DỰNG FACT_CUSTOMER_METRICS ===
  -> Snapshot Date Key: 20221231 (2022-12-31)
✓ Xây dựng Fact_Customer_Metrics hoàn tất! Tổng số khách hàng có đơn giao thành công: 85,115


,Customer_Metric_Key,Customer_ID,Snapshot_Date_Key,Zip_Code,First_Order_Date,Last_Order_Date,Total_Delivered_Orders,Total_Lifetime_Revenue,Total_Lifetime_Items,Lifespan_Days,Inter_Purchase_Days,Customer_AOV,Customer_Basket_Size,Is_Repeat_Customer,Customer_Type,CLV_Rank,Customer_Tier
0,1,1,20221231,15201,2012-07-25,2020-02-24,4,98782.44,22,2770,923.33,24695.61,5.50,1,Repeat Customer,37432,Potential Customers
1,2,2,20221231,15201,2013-09-20,2022-07-06,2,126255.29,14,3211,3211.00,63127.64,7.00,1,Repeat Customer,31668,Potential Customers
2,3,3,20221231,15201,2012-08-27,2012-11-24,2,12001.55,4,89,89.00,6000.78,2.00,1,Repeat Customer,74824,Low-value Customers
3,4,4,20221231,15201,2020-06-28,2020-06-28,1,10939.06,8,0,0.00,10939.06,8.00,0,One-time Buyer,75781,Low-value Customers
4,5,5,20221231,15201,2012-08-09,2019-03-27,5,64179.86,23,2421,605.25,12835.97,4.60,1,Repeat Customer,47776,Low-value Customers


## 📊 BƯỚC 5: Kiểm tra Chất lượng Dữ liệu & Xuất File CSV / ZIP

In [5]:
print("=================================================================")
print("             BÁO CÁO KIỂM TRA CHẤT LƯỢNG 2 BẢNG FACT             ")
print("=================================================================\n")

facts_dict = {
    'Fact_Sales_Order_Items': fact_sales,
    'Fact_Customer_Metrics': fact_cust_metrics
}

for name, df_fact in facts_dict.items():
    pk_col = df_fact.columns[0]
    pk_nulls = df_fact[pk_col].isnull().sum()
    pk_dups = df_fact[pk_col].duplicated().sum()

    status_pk = "PASSED ✓" if (pk_nulls == 0 and pk_dups == 0) else f"FAILED ✗ ({pk_dups} dups, {pk_nulls} nulls)"

    print(f"📦 Bảng [{name}]:")
    print(f"   • Tổng số dòng        : {len(df_fact):,}")
    print(f"   • Số lượng cột        : {len(df_fact.columns)}")
    print(f"   • Khóa chính ({pk_col}): {status_pk}")

    # 1. Xuất file vào thư mục Gold_Facts (cho Fact standalone)
    output_file_facts = os.path.join(OUTPUT_DIR, f"{name}.csv")
    df_fact.to_csv(output_file_facts, index=False, encoding='utf-8-sig')
    print(f"   • Đã lưu CSV (Gold_Facts)      : '{output_file_facts}'")

    # 2. Xuất file vào thư mục Gold_Data_Warehouse (cho script Bulk Insert SSMS)
    output_file_dw = os.path.join(OUTPUT_DW_DIR, f"{name}.csv")
    df_fact.to_csv(output_file_dw, index=False, encoding='utf-8-sig')
    print(f"   • Đã lưu CSV (Gold_DW)         : '{output_file_dw}'\n")

# Nén thư mục kết quả
zip_filename = 'Gold_Facts'
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)
print(f"✓ Đã nén toàn bộ bảng Fact thành file: '{zip_filename}.zip'")

# Tải về máy nếu trên Colab
try:
    from google.colab import files
    print("\n-> Đang tự động tải file 'Gold_Facts.zip' về máy tính của bạn...")
    files.download(f"{zip_filename}.zip")
except ImportError:
    print(f"\n-> Môi trường Local: Toàn bộ file đã được lưu tại thư mục '{OUTPUT_DIR}/', '{OUTPUT_DW_DIR}/' và '{zip_filename}.zip'")

print("\n=================================================================")
print("🎉 HOÀN THÀNH ETL TOÀN BỘ 2 BẢNG FACT (NATURAL KEYS)!           ")
print("=================================================================")

             BÁO CÁO KIỂM TRA CHẤT LƯỢNG 2 BẢNG FACT             

📦 Bảng [Fact_Sales_Order_Items]:
   • Tổng số dòng        : 714,653
   • Số lượng cột        : 17
   • Khóa chính (Sales_Item_Key): PASSED ✓
   • Đã lưu CSV (Gold_Facts)      : 'Gold_Facts/Fact_Sales_Order_Items.csv'
   • Đã lưu CSV (Gold_DW)         : 'Gold_Data_Warehouse/Fact_Sales_Order_Items.csv'

📦 Bảng [Fact_Customer_Metrics]:
   • Tổng số dòng        : 85,115
   • Số lượng cột        : 17
   • Khóa chính (Customer_Metric_Key): PASSED ✓
   • Đã lưu CSV (Gold_Facts)      : 'Gold_Facts/Fact_Customer_Metrics.csv'
   • Đã lưu CSV (Gold_DW)         : 'Gold_Data_Warehouse/Fact_Customer_Metrics.csv'

✓ Đã nén toàn bộ bảng Fact thành file: 'Gold_Facts.zip'

-> Đang tự động tải file 'Gold_Facts.zip' về máy tính của bạn...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 HOÀN THÀNH ETL TOÀN BỘ 2 BẢNG FACT (NATURAL KEYS)!           
